# FlashNystrom vs the sub-quadratic field: MQAR (Zoology figure-2 protocol)

Apples-to-apples multi-query associative recall. Every mixer runs in ONE
environment, same backbone / data / protocol; only the sequence-mixer operator
changes. Backends: `sdpa`, `linear_attention`, `nystrom_reference`,
`flash_nystrom`, `flash_nystrom_tc`, `hyena`, `mamba`.

## Protocol: HazyResearch/zoology, `iclr24_zoology_figure2/configs.py`

This notebook follows the config behind the published MQAR comparison
(Arora et al., ICLR 2024), read from the official source rather than reproduced
from memory:

| Setting | Value | Source |
|---|---|---|
| Training data | **fixed set, generated once, reused every epoch** | `train.py:242`, `data/utils.py:85-89,117` |
| Train / test | 100,000 / 3,000 | `configs.py:39-40` |
| LR sweep | `np.logspace(-4,-2,4)` | `configs.py:52` |
| d_model | swept 64 / 128 / 256 / 512 | `configs.py:46-51` |
| Optimizer | flat AdamW, wd=0.1, **no** param groups | `train.py:185-189` |
| Grad clipping | none | `train.py:117-144` |
| Scheduler | CosineAnnealingLR, per-epoch, no warmup | `train.py:190-192,207` |
| Layers | 2, every layer the same mixer; no MLP | `configs.py:140,145`, `model.py:243` |
| Position embeddings | attention only | `configs.py:142` |
| `random_non_queries` | True (see below) | `multiquery_ar.py:12` |
| Early stop | test accuracy > 0.99 | `config.py:133-134` |
| seq_len / kv, batch, epochs, vocab | 256 / 16, 256, 64, 8192 | `configs.py` |

**The recall-vs-`d_model` curve IS the result.** Zoology's finding is that gated
convolutions and SSMs need a far larger model dimension than attention to solve
MQAR. A single number at one `d_model` is one point on that curve, not the
benchmark, so this sweeps all four dimensions.

**Non-query slots carry random tokens** (`random_non_queries=True`, Zoology's
`MQARConfig` default; their figure-2 config overrides it to False). This is the
harder setting and the one that separates the operators. With blank slots the
task saturates near ceiling within a couple of epochs, which would leave every
method tied and the comparison uninformative.

## Two forced deviations, applied uniformly to every backend

1. **Bidirectional mixing + predict-in-place labels** (Zoology is causal /
   next-token). FlashNystrom has no causal kernel. MQAR stays solvable: every
   query's bound value lies earlier in the sequence, so dropping the causal mask
   leaks nothing.
2. **bf16, not fp32.** The FlashNystrom kernel exhausts shared memory in fp32
   (`kernel3_scalar: insufficient smem`), so fp32 is unavailable to our own
   method. Every baseline runs in the same bf16 so none is advantaged.

Both are disclosed in the paper. Zoology uses ONE seed per configuration
(`config.py:139`), which is what `N_SEEDS=1` reproduces; raise it for error bars,
at linear cost.

In [ ]:
import torch, subprocess, sys

def run_streaming(cmd):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end=""); sys.stdout.flush()
    p.wait()
    return p.returncode

print(subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
CC = torch.cuda.get_device_capability()
print("compute capability:", CC, "| flash_nystrom + mamba-ssm supported:", CC >= (8, 0))

In [ ]:
# Clone (with the CUTLASS submodule the kernel build needs), compile flash_nystrom
# for this arch, and install the real Mamba CUDA kernels.
import os, torch
%cd /content
!rm -rf FlashNystrom
!git clone --recursive -q https://github.com/athrva98/FlashNystrom.git
%cd /content/FlashNystrom
!pip -q install einops
CC = torch.cuda.get_device_capability()
if CC >= (8, 0):
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{CC[0]}.{CC[1]}"
    os.environ["FLASH_NYSTROM_LAX_BUILD"] = "1"  # tolerate 3rd-party header warnings
    !pip install -e . --no-build-isolation
    # Real Mamba kernels. Their setup.py imports torch, so pip build isolation
    # fails at 'getting requirements to build wheel'; --no-build-isolation uses the
    # torch already installed. causal-conv1d first (mamba-ssm depends on it).
    !pip -q install ninja packaging
    !pip install causal-conv1d --no-build-isolation
    !pip install mamba-ssm --no-build-isolation
    import flash_nystrom
    print("flash_nystrom built, version", flash_nystrom.__version__)
    from paper.mqar.baselines import _HAS_MAMBA_CUDA
    print("Mamba CUDA kernels available:", _HAS_MAMBA_CUDA)
else:
    print("sm < 8.0: skipping kernel builds; pure-PyTorch backends only (Mamba slow)")

In [ ]:
# Zoology figure-2 sweep: backend x d_model x lr. Resumable (skips existing JSONs).
import os, numpy as np, torch

N_SEEDS   = 1                              # Zoology uses one seed (config.py:139)
LR_GRID   = list(np.logspace(-4, -2, 4))   # configs.py:52
D_MODELS  = [64, 128, 256, 512]            # configs.py:46-51
LAYOUT    = "uniform"                      # every layer is the mixer (figure 2)
BACKENDS  = ["sdpa", "linear_attention", "nystrom_reference",
             "flash_nystrom", "flash_nystrom_tc", "hyena", "mamba"]
if torch.cuda.get_device_capability() < (8, 0):
    BACKENDS = [b for b in BACKENDS if "flash_nystrom" not in b]

OUT = "runs/mqar_fig2"
os.makedirs(OUT, exist_ok=True)

def heads_for(d):
    """head_dim must be 64 or 128 for the FlashNystrom kernel; this keeps it at
    128 for d>=128 and 64 at d=64, so every backend sees the same head shape."""
    return max(1, d // 128)

total = len(BACKENDS) * len(D_MODELS) * len(LR_GRID) * N_SEEDS
print(f"{total} runs = {len(BACKENDS)} backends x {len(D_MODELS)} d_model "
      f"x {len(LR_GRID)} lr x {N_SEEDS} seed(s)")
print("NOTE: 100k train examples x 64 epochs = ~25k steps per run (early stop at "
      "0.99 cuts the easy ones). This is a multi-hour sweep; it is resumable, so "
      "re-running this cell after a disconnect continues where it left off.")

for b in BACKENDS:
    for d in D_MODELS:
        for lr in LR_GRID:
            for s in range(N_SEEDS):
                out = f"{OUT}/{b}_d{d}_lr{lr:.2e}_seed{s}.json"
                if os.path.exists(out):
                    print("skip", out); continue
                print(f"\n===== {b}  d_model={d}  lr={lr:.3e}  seed {s} =====", flush=True)
                run_streaming(["python", "-u", "-m", "paper.mqar.train",
                    "--backend", b, "--seed", str(s), "--dim", str(d),
                    "--heads", str(heads_for(d)), "--lr", f"{lr:.6e}",
                    "--layer_layout", LAYOUT, "--random_non_queries",
                    "--kappa_star", "0", "--seq_len", "256", "--num_kv_pairs", "16",
                    "--num_landmarks", "64", "--newton_iter", "6",
                    "--batch_size", "256", "--epochs", "64",
                    "--num_train", "100000", "--num_test", "3000",
                    "--out_json", out])

In [ ]:
# Aggregate: recall vs d_model, best over the LR grid (Zoology's reporting).
import json, glob, statistics as st
from collections import defaultdict

runs = defaultdict(list)   # (backend, d_model, lr) -> [recall]
for f in glob.glob("runs/mqar_fig2/*.json"):
    r = json.load(open(f))
    runs[(r["backend"], r["dim"], r["lr"])].append(r["best_recall"])

order = ["sdpa", "linear_attention", "nystrom_reference",
         "flash_nystrom", "flash_nystrom_tc", "hyena", "mamba"]
dims = sorted({d for (_, d, _) in runs})

print("MQAR recall vs model dimension (best over LR grid, Zoology figure 2):")
print(f"  {'backend':<20} " + "  ".join(f"d={d:<6}" for d in dims))
for b in order:
    row = []
    for d in dims:
        vals = [st.mean(v) for (bb, dd, _), v in runs.items() if bb == b and dd == d]
        row.append(f"{max(vals):8.2f}" if vals else f"{'--':>8}")
    if any("-" not in c for c in row):
        print(f"  {b:<20} " + "  ".join(row))

print("\nBest LR per (backend, d_model):")
for b in order:
    for d in dims:
        cfg = [(lr, st.mean(v)) for (bb, dd, lr), v in runs.items() if bb == b and dd == d]
        if cfg:
            lr_b, r_b = max(cfg, key=lambda x: x[1])
            print(f"  {b:<20} d={d:<5} lr={lr_b:.2e}  recall={r_b:6.2f}%")